# Financial Q&A 10-K: LLM-Based Financial Question Answering Pipeline

This notebook builds a lightweight LLM-based question-answering pipeline using 50 samples from the [Financial Q&A 10-K dataset (Kaggle)](https://www.kaggle.com/datasets/yousefsaeedian/financial-q-and-a-10k).

We compare two models that are **gpt-5.6-sol** and **gpt-5.6-luna** to evaluate tradeoffs between answer quality, consistency, abstention behaviour, and cost in a financial QA context.

## Pipeline Overview:

1. **Data Loading & Validation**
2. **Sample Selection**
3. **Prompt Design** 
4. **Model Selection** 
5. **Answer Generation**
6. **Evaluation** 
7. **Model Comparison & Recommendation** 

## Assumptions & Provided Constraints:

- Answers are grounded **only** in the provided context — no external retrieval (RAG) or outside knowledge or (MCP) tool will be used
- The first 50 samples are selected for simplicity and reproducibility
- Evaluation prioritises soundness and interpretability over metric volume

## 1. Data Loading & Validation:

In [ ]:
import os
import sys
import re
import json
import time
import tiktoken
import pandas as pd
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv

print(f"Python: {sys.version.split()[0]}")
print(f"Pandas: {pd.__version__}")

Python: 3.12.8
Pandas: 3.0.5


In [92]:
RAW_DATA_PATH = "/Users/yagyansh/Desktop/pwc-take-home-task/data/Financial-QA-10k.csv"

raw_df = pd.read_csv(RAW_DATA_PATH)

print("Shape:", raw_df.shape)
print("Columns:", raw_df.columns.tolist())

Shape: (7000, 5)
Columns: ['question', 'answer', 'context', 'ticker', 'filing']


In [93]:
raw_df.head()

,question,answer,context,ticker,filing
0,What area did NVIDIA initially focus on before...,NVIDIA initially focused on PC graphics.,"Since our original focus on PC graphics, we ha...",NVDA,2023_10K
1,What are some of the recent applications of GP...,Recent applications of GPU-powered deep learni...,Some of the most recent applications of GPU-po...,NVDA,2023_10K
2,What significant invention did NVIDIA create i...,NVIDIA invented the GPU in 1999.,Our invention of the GPU in 1999 defined moder...,NVDA,2023_10K
3,How does NVIDIA's platform strategy contribute...,NVIDIA's platform strategy brings together har...,"NVIDIA has a platform strategy, bringing toget...",NVDA,2023_10K
4,What does NVIDIA's CUDA programming model enable?,NVIDIA's CUDA programming model opened the par...,With our introduction of the CUDA programming ...,NVDA,2023_10K


In [94]:
raw_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7000 entries, 0 to 6999
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   question  6998 non-null   str  
 1   answer    6998 non-null   str  
 2   context   6999 non-null   str  
 3   ticker    7000 non-null   str  
 4   filing    7000 non-null   str  
dtypes: str(5)
memory usage: 273.6 KB


## 2. Sample Selection:

The assignment requires exactly 50 samples. Before selecting, I drop rows with missing values in the core fields required for question answering(`question`, `reference_answer`, `context`). This ensures every sample is complete and usable for both generation and evaluation.

The first 50 valid entries are then selected for reproducibility.

In [ ]:
required_columns = ["question", "context", "answer"]
valid_df = raw_df.dropna(subset=required_columns).copy()

print(f"Valid rows after dropping nulls: {len(valid_df)}")

samples_50 = valid_df.head(50).copy()
assert len(samples_50) == 50, f"Expected 50 samples, got {len(samples_50)}"
print(f"Selected samples: {samples_50.shape}")
samples_50.head()

Valid rows after dropping nulls: 6997
Selected samples: (50, 5)


,question,answer,context,ticker,filing
0,What area did NVIDIA initially focus on before...,NVIDIA initially focused on PC graphics.,"Since our original focus on PC graphics, we ha...",NVDA,2023_10K
1,What are some of the recent applications of GP...,Recent applications of GPU-powered deep learni...,Some of the most recent applications of GPU-po...,NVDA,2023_10K
2,What significant invention did NVIDIA create i...,NVIDIA invented the GPU in 1999.,Our invention of the GPU in 1999 defined moder...,NVDA,2023_10K
3,How does NVIDIA's platform strategy contribute...,NVIDIA's platform strategy brings together har...,"NVIDIA has a platform strategy, bringing toget...",NVDA,2023_10K
4,What does NVIDIA's CUDA programming model enable?,NVIDIA's CUDA programming model opened the par...,With our introduction of the CUDA programming ...,NVDA,2023_10K


In [96]:
OUTPUT_PATH = "/Users/yagyansh/Desktop/pwc-take-home-task/data/samples_50.csv"

samples_50.to_csv(OUTPUT_PATH, index = False)
print(f"Saved samples to {OUTPUT_PATH}")

Saved samples to /Users/yagyansh/Desktop/pwc-take-home-task/data/samples_50.csv


In [97]:
check_df = pd.read_csv(OUTPUT_PATH)

print("Verified shapeL", check_df.shape)
print("Columns:", check_df.columns.tolist())

Verified shapeL (50, 5)
Columns: ['question', 'answer', 'context', 'ticker', 'filing']


## 3. Prompt Design:
The model receives the **question** and the supporting **context** as input. The reference answer (`answer` column) is deliberately excluded from the prompt as it serves solely as ground truth for evaluation.

The system prompt instructs the model to:
- Answer **only** from the provided context (no external knowledge)
- Provide a **confidence score** (0.0–1.0) reflecting certainty in the answer
- **Abstain** when the context does not sufficiently support a clear answer
- Return responses in a **structured CSV format** for downstream processing


Ran into issues with Prompt Designing initially but used **CRISP** format to get the best out of these systems.

In [ ]:
SYSTEM_PROMPT = """You are a financial question-answering assistant specialising in 10-K filings.

Rules:
- Answer using ONLY the information in the provided context.
- Do not use external knowledge or information from your training data.
- Do not make assumptions unsupported by the context.
- Do not invent facts or figures.
- Keep the answer concise and directly address the question.
- Provide a confidence score between 0.0 and 1.0 reflecting how well the context supports your answer.
- If the context does not contain sufficient information to answer reliably, respond with exactly: ABSTAIN"""


def build_user_prompt(question: str, context: str) -> str:
    """Construct the user-facing prompt with context and question."""
    return f"""Context:
{context}

Question:
{question}

Provide your answer in the following format:
Answer: <your answer>
Confidence: <0.0 to 1.0>"""

In [ ]:
test_sample = samples_50.iloc[0]

user_prompt = build_user_prompt(test_sample["question"], test_sample["context"])
print(user_prompt)

print(f"\nQuestion length:  {len(test_sample['question']):,} chars")
print(f"Context length:   {len(test_sample['context']):,} chars")
print(f"Reference answer: {test_sample['answer'][:200]}...")

Context:
Since our original focus on PC graphics, we have expanded to several other large and important computationally intensive fields.

Question:
What area did NVIDIA initially focus on before expanding to other computationally intensive fields?

Provide your answer in the following format:
Answer: <your answer>
Confidence: <0.0 to 1.0>

Question length:  99 chars
Context length:   128 chars
Reference answer: NVIDIA initially focused on PC graphics....


In [11]:
print("Question Length:", len(test_sample["question"]))
print("Context Length:", len(test_sample["context"]))
print("Reference Answer Length:", len(test_sample["answer"]))

Question Length: 99
Context Length: 128
Reference Answer Length: 40


## 4. Model Selection:

Two models from the OpenAI family are selected to evaluate the tradeoff between answer quality and cost efficiency. Both models are evaluated using the same 50 samples and identical prompt structure to ensure a fair comparison.

| | Model | API ID | Rationale |
|---|---|---|---|
| **Model A** | GPT-5.6 Sol | `gpt-5.6-sol` | Higher-capability model; prioritises answer quality and reliability |
| **Model B** | GPT-5.6 Luna | `gpt-5.6-luna` | Lighter-weight model; prioritises speed and cost efficiency |

### Comparison Dimensions

- **Answer correctness** — How accurately does the model answer relative to the reference?
- **Context adherence** — Does the model stay grounded in the provided context?
- **Abstention behaviour** — Does the model appropriately abstain when context is insufficient?
- **Consistency** — Are outputs stable and well-structured across samples?
- **Cost** — What is the per-query cost tradeoff relative to quality?

Both models receive only the `question` and `context` fields — the reference answer is withheld entirely.

In [109]:
load_dotenv(dotenv_path=".env")  

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found. Ensure a .env file exists with your key.")

client = OpenAI(api_key=api_key)
print("OpenAI client initialised successfully.")

OpenAI client initialised successfully.


In [ ]:
# models = client.models.list()
# for model in models.data:
#     if any(name in model.id.lower() for name in ["gpt-5", "gpt-4.1", "gpt-4o"]):
#         print(model.id)

In [ ]:

MODEL_A = "gpt-5.6-sol"    # this one is for quality
MODEL_B = "gpt-5.6-luna"   # this one is more cost oriented

print(f"Model A: {MODEL_A}")
print(f"Model B: {MODEL_B}")

Model A: gpt-5.6-sol
Model B: gpt-5.6-luna


## 5. Answer Generation — Test Run:

Before running the full pipeline across all 50 samples, a single-sample test verifies that the API call, prompt structure, and response parsing work as expected.

In [ ]:
test_sample = samples_50.iloc[0]

test_prompt = build_user_prompt(
    test_sample["question"],
    test_sample["context"]
)

print("QUESTION:")
print(test_sample["question"])
print(f"\nREFERENCE ANSWER (which we will being withholding from model):")
print(test_sample["answer"])

QUESTION:
What area did NVIDIA initially focus on before expanding to other computationally intensive fields?

REFERENCE ANSWER (which we will being withholding from model):
NVIDIA initially focused on PC graphics.


## Model A:

In [ ]:
response_a = client.responses.create(
    model=MODEL_A,
    instructions=SYSTEM_PROMPT,
    input=test_prompt,
    reasoning={"effort": "none"} 
)

model_a_answer = response_a.output_text

print("MODEL A ANSWER:")
print(model_a_answer)

MODEL A ANSWER:
Answer: PC graphics
Confidence: 1.0


## Model B:

In [ ]:
response_b = client.responses.create(
    model=MODEL_B,
    instructions=SYSTEM_PROMPT,
    input=test_prompt,
    reasoning={"effort": "none"} 
)

model_b_answer = response_b.output_text

print("MODEL B ANSWER:")
print(model_b_answer)

MODEL B ANSWER:
Answer: PC graphics  
Confidence: 1.0


## 5. Answer Generation - Pipeline:

In [ ]:
def generate_answer(model_name: str, question: str, context: str) -> dict:
    """Generate an answer from the given model using only the provided context."""
    prompt = build_user_prompt(question, context)

    try:
        response = client.responses.create(
            model=model_name,
            instructions=SYSTEM_PROMPT,
            input=prompt,
            reasoning={"effort": "none"}
        )
        raw_output = response.output_text.strip()

        
        answer, confidence = _parse_response(raw_output)

        return {
            "predicted_answer": answer,
            "confidence": confidence,
            "abstained": answer.upper() == "ABSTAIN",
            "response_id": response.id,
            "error": None
        }

    except Exception as e:
        return {
            "predicted_answer": None,
            "confidence": None,
            "abstained": True,
            "response_id": None,
            "error": str(e)
        }


def _parse_response(raw_output: str) -> tuple:
    """Extract answer and confidence from structured model output."""
    answer = raw_output
    confidence = None

    for line in raw_output.split("\n"):
        if line.lower().startswith("answer:"):
            answer = line.split(":", 1)[1].strip()
        elif line.lower().startswith("confidence:"):
            try:
                confidence = float(line.split(":", 1)[1].strip())
            except ValueError:
                confidence = None

    return answer, confidence

In [ ]:
test_result_a = generate_answer(MODEL_A, test_sample["question"], test_sample["context"])
test_result_b = generate_answer(MODEL_B, test_sample["question"], test_sample["context"])

print(f"MODEL A ({MODEL_A}):")
for k, v in test_result_a.items():
    print(f"  {k}: {v}")

print(f"\nMODEL B ({MODEL_B}):")
for k, v in test_result_b.items():
    print(f"  {k}: {v}")

MODEL A (gpt-5.6-sol):
  predicted_answer: PC graphics
  confidence: 1.0
  abstained: False
  response_id: resp_06d5f09d6ad6eb23006a9ea8b0923087d2954a19f70ef43ca7
  error: None

MODEL B (gpt-5.6-luna):
  predicted_answer: PC graphics
  confidence: 1.0
  abstained: False
  response_id: resp_04e4c46c240cbcd0006a9ea8b3bae087d28a2629de6f111df3
  error: None


In [ ]:
def run_model_on_samples(model_name: str, samples: pd.DataFrame) -> pd.DataFrame:
    """Run a model across all samples and return structured predictions."""
    results = []
    start_time = time.time()

    for i, (idx, row) in enumerate(samples.iterrows(), 1):
        result = generate_answer(model_name, row["question"], row["context"])

        results.append({
            "sample_id": i,
            "question": row["question"],
            "context": row["context"],
            "reference_answer": row["answer"],
            "predicted_answer": result["predicted_answer"],
            "confidence": result["confidence"],
            "abstained": result["abstained"],
            "response_id": result["response_id"],
            "error": result["error"],
            "model": model_name
        })

        if i % 10 == 0 or i == len(samples):
            elapsed = time.time() - start_time
            print(f"  {model_name}: {i}/{len(samples)} completed ({elapsed:.1f}s)")

        time.sleep(0.3) 

    print(f"  {model_name}: Done — {len(results)} predictions in {time.time() - start_time:.1f}s")
    return pd.DataFrame(results)

In [ ]:
print(f"Running {MODEL_A} on {len(samples_50)} samples...\n")
results_a = run_model_on_samples(MODEL_A, samples_50)

Running gpt-5.6-sol on 50 samples...

  gpt-5.6-sol: 10/50 completed (24.4s)
  gpt-5.6-sol: 20/50 completed (48.0s)
  gpt-5.6-sol: 30/50 completed (71.5s)
  gpt-5.6-sol: 40/50 completed (96.4s)
  gpt-5.6-sol: 50/50 completed (118.6s)
  gpt-5.6-sol: Done — 50 predictions in 118.9s


In [ ]:
print(f"Running {MODEL_B} on {len(samples_50)} samples...\n")
results_b = run_model_on_samples(MODEL_B, samples_50)

Running gpt-5.6-luna on 50 samples...

  gpt-5.6-luna: 10/50 completed (21.0s)
  gpt-5.6-luna: 20/50 completed (39.1s)
  gpt-5.6-luna: 30/50 completed (57.6s)
  gpt-5.6-luna: 40/50 completed (73.4s)
  gpt-5.6-luna: 50/50 completed (89.6s)
  gpt-5.6-luna: Done — 50 predictions in 89.9s


In [ ]:
results_a[["sample_id", "reference_answer", "predicted_answer", "confidence", "abstained"]].head(10)

,sample_id,reference_answer,predicted_answer,confidence,abstained
0,1,NVIDIA initially focused on PC graphics.,PC graphics,1.0,False
1,2,Recent applications of GPU-powered deep learni...,Recent applications include recommendation sys...,1.0,False
2,3,NVIDIA invented the GPU in 1999.,The GPU (graphics processing unit),1.0,False
3,4,NVIDIA's platform strategy brings together har...,NVIDIA’s platform strategy integrates hardware...,1.0,False
4,5,NVIDIA's CUDA programming model opened the par...,It enables the GPU’s parallel processing capab...,1.0,False
5,6,NVIDIA's GPUs and software are used for automa...,"Transportation, healthcare, financial services...",1.0,False
6,7,NVIDIA and SoftBank terminated their Share Pur...,They terminated the agreement because signific...,1.0,False
7,8,NVIDIA recorded an acquisition termination cos...,$1.35 billion,1.0,False
8,9,The NVIDIA computing platform focuses on accel...,"The most compute-intensive workloads, includin...",1.0,False
9,10,The NVIDIA computing platform includes energy-...,The NVIDIA computing platform consists of ener...,1.0,False


In [ ]:
results_a['confidence'].value_counts()

confidence
1.00    46
0.98     3
0.99     1
Name: count, dtype: int64

In [ ]:
results_b[["sample_id", "reference_answer", "predicted_answer", "confidence", "abstained"]].head(10)

,sample_id,reference_answer,predicted_answer,confidence,abstained
0,1,NVIDIA initially focused on PC graphics.,PC graphics,1.0,False
1,2,Recent applications of GPU-powered deep learni...,"Recommendation systems, large language models,...",1.0,False
2,3,NVIDIA invented the GPU in 1999.,The GPU (graphics processing unit).,1.0,False
3,4,NVIDIA's platform strategy brings together har...,"NVIDIA’s platform strategy combines hardware, ...",1.0,False
4,5,NVIDIA's CUDA programming model opened the par...,It enables general-purpose computing using the...,1.0,False
5,6,NVIDIA's GPUs and software are used for automa...,"Transportation, healthcare, financial services...",1.0,False
6,7,NVIDIA and SoftBank terminated their Share Pur...,They terminated the agreement because signific...,1.0,False
7,8,NVIDIA recorded an acquisition termination cos...,$1.35 billion,1.0,False
8,9,The NVIDIA computing platform focuses on accel...,The NVIDIA computing platform focuses on accel...,1.0,False
9,10,The NVIDIA computing platform includes energy-...,"Energy-efficient GPUs, data processing units (...",1.0,False


In [ ]:
results_b['confidence'].value_counts()

confidence
1.00    48
0.98     2
Name: count, dtype: int64

## 6. Evaluation

Predictions are evaluated against the reference answers using a combination of quantitative metrics. Each metric is chosen to capture a different dimension of answer quality:

- **ROUGE-L** — Measures longest common subsequence overlap between predicted and reference answers. Captures structural similarity and recall of key phrases.
- **Semantic Similarity** (cosine similarity via embeddings) — Measures meaning-level alignment, accounting for paraphrasing the model may produce.
- **Exact Match** — Strict binary check. Useful as a lower bound but expected to be low given the free-text nature of financial answers.
- **Abstention Rate** — Proportion of samples where the model chose not to answer. A healthy abstention rate signals appropriate uncertainty calibration.

These metrics are selected to balance surface-level overlap (ROUGE, Exact Match) with semantic understanding (embedding similarity), while also evaluating the model's ability to know when it *doesn't* know (abstention).

In [ ]:
def normalize_text(text: str) -> str:
    if text is None:
        return ""

    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)       
    text = re.sub(r"[^\w\s.%$-]", "", text) 

    return text

In [ ]:
def exact_match(predicted: str, reference: str) -> bool:
    return normalize_text(predicted) == normalize_text(reference)

In [ ]:
for df, name in [(results_a, MODEL_A), (results_b, MODEL_B)]:
    df["exact_match"] = df.apply(
        lambda row: exact_match(row["predicted_answer"], row["reference_answer"]),
        axis=1
    )
    print(f"{name} — Exact Match Rate: {df['exact_match'].mean():.1%}")

gpt-5.6-sol — Exact Match Rate: 8.0%
gpt-5.6-luna — Exact Match Rate: 8.0%


> **Note:** Initially considered using ROUGE-L for evaluation but found it unreliable 

> for short financial answers where word overlap is low but meaning is preserved. 

> Replaced with LLM-as-a-Judge for semantic evaluation.

### LLM-As-A-Judge:

Traditional metrics like exact match and ROUGE capture surface-level overlap but struggle with semantically correct answers that are phrased differently from the reference. To address this, an LLM is used as a judge to score each prediction on a 1–5 scale.

The judge model evaluates:
- **Correctness** — Does the predicted answer convey the same information as the reference?
- **Completeness** — Does it capture the key details without significant omission?
- **Context adherence** — Is the answer grounded in the provided context?

This approach is well-suited to financial QA, where a predicted answer like *"revenue increased by 12%"* should score highly against a reference of *"there was a 12% rise in revenue"*, even though exact match would return `False`.

> **Note:** The judge model is a separate LLM call and incurs additional API cost. This is a deliberate tradeoff for higher-quality evaluation.

> **Note:** Personal opinion based on my exposure and building with tools; LLM As A Judge is not always reliable; however for this assignment am using it.

In [ ]:
JUDGE_MODEL = "gpt-5.6-sol" 

def build_evaluation_prompt(
    question: str,
    context: str,
    reference_answer: str,
    predicted_answer: str
) -> str:
    """Build the prompt for LLM-based evaluation of a predicted answer."""
    return f"""You are evaluating a financial question-answering system.

Your task is to determine whether the predicted answer is correct based ONLY
on the provided question, context, and reference answer.

Evaluate the predicted answer using these criteria:

- CORRECT: The prediction conveys the same substantive answer as the reference
  and is supported by the context. Minor phrasing differences are acceptable.
- INCORRECT: The prediction contains a materially wrong answer, unsupported
  information, or contradicts the reference or context.
- ABSTAINED: The prediction explicitly states ABSTAIN or indicates that there
  is insufficient information to answer.

Question:
{question}

Context:
{context}

Reference Answer:
{reference_answer}

Predicted Answer:
{predicted_answer}

Return exactly one label: CORRECT, INCORRECT, or ABSTAINED"""

In [ ]:
JUDGE_MODEL = "gpt-4o-mini" 

In [ ]:
evaluation_sample = results_a.iloc[0]

evaluation_prompt = build_evaluation_prompt(
    evaluation_sample["question"],
    evaluation_sample["context"],
    evaluation_sample["reference_answer"],
    evaluation_sample["predicted_answer"]
)

judge_response = client.responses.create(
    model=JUDGE_MODEL,
    input=evaluation_prompt
)

judge_result = judge_response.output_text.strip().upper()

print(f"Predicted Answer: {evaluation_sample['predicted_answer'][:100]}...")
print(f"Reference Answer: {evaluation_sample['reference_answer'][:100]}...")
print(f"Judge Verdict:    {judge_result}")

Predicted Answer: PC graphics...
Reference Answer: NVIDIA initially focused on PC graphics....
Judge Verdict:    CORRECT


In [ ]:
def evaluate_prediction(
    question: str,
    context: str,
    reference_answer: str,
    predicted_answer: str
) -> str:
    """Use LLM judge to classify a prediction as CORRECT, INCORRECT, or ABSTAINED."""
    
    
    if not predicted_answer or predicted_answer.strip() == "":
        return "ABSTAINED"

    prompt = build_evaluation_prompt(question, context, reference_answer, predicted_answer)

    valid_labels = {"CORRECT", "INCORRECT", "ABSTAINED"}

    try:
        response = client.responses.create(
            model=JUDGE_MODEL,
            input=prompt
        )
        result = response.output_text.strip().upper()

        if result not in valid_labels:
            for label in valid_labels:
                if label in result:
                    return label
            return "UNKNOWN"

        return result

    except Exception as e:
        print(f"  Judge error: {e}")
        return "UNKNOWN"

In [ ]:
sample = results_a.iloc[0]

test_eval_a = evaluate_prediction(
    sample["question"],
    sample["context"],
    sample["reference_answer"],
    sample["predicted_answer"]
)

print(f"Predicted: {sample['predicted_answer'][:100]}...")
print(f"Reference: {sample['reference_answer'][:100]}...")
print(f"Judge Verdict: {test_eval_a}")

Predicted: PC graphics...
Reference: NVIDIA initially focused on PC graphics....
Judge Verdict: CORRECT


In [ ]:
for df, name in [(results_a, MODEL_A), (results_b, MODEL_B)]:
    print(f"Judging {name}...")
    df["judge_verdict"] = df.apply(
        lambda row: evaluate_prediction(
            row["question"],
            row["context"],
            row["reference_answer"],
            row["predicted_answer"]
        ),
        axis=1
    )
    print(f"  {name} — {df['judge_verdict'].value_counts().to_dict()}\n")

Judging gpt-5.6-sol...
  gpt-5.6-sol — {'CORRECT': 50}

Judging gpt-5.6-luna...
  gpt-5.6-luna — {'CORRECT': 47, 'INCORRECT': 3}



In [ ]:
summary = pd.DataFrame({
    MODEL_A: results_a["judge_verdict"].value_counts(),
    MODEL_B: results_b["judge_verdict"].value_counts()
}).fillna(0).astype(int)

summary

,gpt-5.6-sol,gpt-5.6-luna
judge_verdict,,
CORRECT,50,47
INCORRECT,0,3


In [ ]:
def calculate_metrics(results: pd.DataFrame) -> dict:
    total = len(results)

    correct = (results["judge_verdict"] == "CORRECT").sum()
    incorrect = (results["judge_verdict"] == "INCORRECT").sum()
    abstained = (results["judge_verdict"] == "ABSTAINED").sum()

    answered = total - abstained

    return {
        "total": total,
        "correct": correct,
        "incorrect": incorrect,
        "abstained": abstained,
        "accuracy": correct / total,
        "answer_accuracy": correct / answered if answered > 0 else 0,
        "abstention_rate": abstained / total,
        "coverage": answered / total
    }

In [ ]:
metrics_a = calculate_metrics(results_a)
metrics_b = calculate_metrics(results_b)

comparison = pd.DataFrame({
    MODEL_A: metrics_a,
    MODEL_B: metrics_b
})


rate_rows = ["accuracy", "answer_accuracy", "abstention_rate", "coverage"]

display_df = comparison.copy().astype(object)  # Allow mixed types

for col in display_df.columns:
    for idx in display_df.index:
        val = comparison.loc[idx, col]
        if idx in rate_rows:
            display_df.loc[idx, col] = f"{val:.1%}"
        else:
            display_df.loc[idx, col] = f"{int(val)}"

display_df

,gpt-5.6-sol,gpt-5.6-luna
total,50,50
correct,50,47
incorrect,0,3
abstained,0,0
accuracy,100.0%,94.0%
answer_accuracy,100.0%,94.0%
abstention_rate,0.0%,0.0%
coverage,100.0%,100.0%


In [51]:
comparison = pd.DataFrame([
    {
        "Model": MODEL_A,
        **metrics_a
    },
    {
        "Model": MODEL_B,
        **metrics_b
    }
])

comparison


,Model,total,correct,incorrect,abstained,accuracy,answer_accuracy,abstention_rate,coverage
0,gpt-5.6-sol,50,49,1,0,0.98,0.98,0.0,1.0
1,gpt-5.6-luna,50,49,1,0,0.98,0.98,0.0,1.0


In [ ]:
for df, name in [(results_a, MODEL_A), (results_b, MODEL_B)]:
    incorrect = df[df["judge_verdict"] == "INCORRECT"][
        ["sample_id", "question", "reference_answer", "predicted_answer"]
    ]
    print(f"\n{'='*60}")
    print(f"{name} — {len(incorrect)} incorrect predictions")
    print(f"{'='*60}")
    display(incorrect)


gpt-5.6-sol — 0 incorrect predictions


,sample_id,question,reference_answer,predicted_answer



gpt-5.6-luna — 3 incorrect predictions


,sample_id,question,reference_answer,predicted_answer
5,6,What industries use NVIDIA's GPUs and software...,NVIDIA's GPUs and software are used for automa...,"Transportation, healthcare, financial services..."
12,13,What does the Bluefield DPU support?,The Bluefield DPU is supported by foundational...,Foundational data-center-infrastructure-on-a-c...
15,16,What generation technology does the 40 Series ...,The 40 Series graphics cards feature third gen...,Third generation RTX technology


In [ ]:
agreement = (
    results_a["predicted_answer"].apply(normalize_text)
    == results_b["predicted_answer"].apply(normalize_text)
)

print(f"Exact normalised agreement: {agreement.sum()} / {len(agreement)}")
print(f"Agreement rate: {agreement.mean():.1%}")

Exact normalised agreement: 14 / 50
Agreement rate: 28.0%


### Human-in-the-Loop Evaluations

Automated evaluation (exact match + LLM judge) may produce false positives or false negatives. To validate the judge's reliability, flagged cases where either model was judged as INCORRECT or ABSTAINED are surfaced for manual inspection.

This step serves as a quality check on the evaluation itself, not just the models. I suppose Human Beings would be needed to steer AI.

In [ ]:
review_df = results_a[
    ["sample_id", "question", "reference_answer", "predicted_answer", "judge_verdict"]
].merge(
    results_b[["sample_id", "predicted_answer", "judge_verdict"]],
    on="sample_id",
    suffixes=("_sol", "_luna")
)

flagged = review_df[
    (review_df["judge_verdict_sol"] != "CORRECT") |
    (review_df["judge_verdict_luna"] != "CORRECT")
]

print(f"Flagged for manual review: {len(flagged)} / {len(review_df)} samples\n")

flagged[
    ["sample_id", "question", "reference_answer",
     "predicted_answer_sol", "judge_verdict_sol",
     "predicted_answer_luna", "judge_verdict_luna"]
]

Flagged for manual review: 3 / 50 samples



,sample_id,question,reference_answer,predicted_answer_sol,judge_verdict_sol,predicted_answer_luna,judge_verdict_luna
5,6,What industries use NVIDIA's GPUs and software...,NVIDIA's GPUs and software are used for automa...,"Transportation, healthcare, financial services...",CORRECT,"Transportation, healthcare, financial services...",INCORRECT
12,13,What does the Bluefield DPU support?,The Bluefield DPU is supported by foundational...,"The BlueField DPU supports software-defined, h...",CORRECT,Foundational data-center-infrastructure-on-a-c...,INCORRECT
15,16,What generation technology does the 40 Series ...,The 40 Series graphics cards feature third gen...,"Third-generation RTX technology, third-generat...",CORRECT,Third generation RTX technology,INCORRECT


### Manual Override:

Upon reviewing flagged cases, three Luna predictions were marked INCORRECT by the judge despite being substantively correct; the answers were shorter or differently phrased but conveyed the same information. This highlights a known limitation of LLM-as-a-Judge: sensitivity to answer verbosity and phrasing style.

Overrides are applied below and propagated back to the source results for corrected final metrics.

In [ ]:
overrides = {
    6:  {"sol": "CORRECT", "luna": "CORRECT"},
    13: {"sol": "CORRECT", "luna": "CORRECT"},
    16: {"sol": "CORRECT", "luna": "CORRECT"},
}

for sample_id, verdicts in overrides.items():
    results_a.loc[results_a["sample_id"] == sample_id, "judge_verdict"] = verdicts["sol"]
    results_b.loc[results_b["sample_id"] == sample_id, "judge_verdict"] = verdicts["luna"]

print(f"Applied {len(overrides)} manual overrides.")
print(f"\nUpdated verdict counts:")
print(f"  {MODEL_A}: {results_a['judge_verdict'].value_counts().to_dict()}")
print(f"  {MODEL_B}: {results_b['judge_verdict'].value_counts().to_dict()}")

Applied 3 manual overrides.

Updated verdict counts:
  gpt-5.6-sol: {'CORRECT': 50}
  gpt-5.6-luna: {'CORRECT': 50}


In [ ]:
def calculate_metrics(results: pd.DataFrame, verdict_col: str = "judge_verdict") -> dict:
    total = len(results)

    correct = (results[verdict_col] == "CORRECT").sum()
    incorrect = (results[verdict_col] == "INCORRECT").sum()
    abstained = (results[verdict_col] == "ABSTAINED").sum()

    answered = correct + incorrect

    return {
        "Total samples": total,
        "Correct": correct,
        "Incorrect": incorrect,
        "Abstained": abstained,
        "Accuracy": correct / total if total else 0,
        "Answer Accuracy": correct / answered if answered else 0,
        "Abstention Rate": abstained / total if total else 0,
        "Coverage": answered / total if total else 0,
    }

In [ ]:
metrics_a_final = calculate_metrics(results_a, "judge_verdict")
metrics_b_final = calculate_metrics(results_b, "judge_verdict")

summary = pd.DataFrame({
    "Metric": list(metrics_a_final.keys()),
    MODEL_A: list(metrics_a_final.values()),
    MODEL_B: list(metrics_b_final.values()),
})

rate_metrics = ["Accuracy", "Answer Accuracy", "Abstention Rate", "Coverage"]
for col in [MODEL_A, MODEL_B]:
    summary[col] = summary.apply(
        lambda row: f"{row[col]:.1%}" if row["Metric"] in rate_metrics else int(row[col]),
        axis=1
    )

summary

,Metric,gpt-5.6-sol,gpt-5.6-luna
0,Total samples,50,50
1,Correct,50,50
2,Incorrect,0,0
3,Abstained,0,0
4,Accuracy,100.0%,100.0%
5,Answer Accuracy,100.0%,100.0%
6,Abstention Rate,0.0%,0.0%
7,Coverage,100.0%,100.0%


In [ ]:
OUTPUT_DIR = "/Users/yagyansh/Desktop/pwc-take-home-task/data"

results_a.to_csv(f"{OUTPUT_DIR}/predictions_model_a.csv", index=False)
results_b.to_csv(f"{OUTPUT_DIR}/predictions_model_b.csv", index=False)
summary.to_csv(f"{OUTPUT_DIR}/evaluation_summary.csv", index=False)

print(f"Saved predictions and evaluation to {OUTPUT_DIR}/")

Saved predictions and evaluation to /Users/yagyansh/Desktop/pwc-take-home-task/data/


## 7. Model Comparison & Recommendation:

### Cost Analysis

Token usage and cost are estimated using `tiktoken` to compare the economic tradeoff between the two models.

In [ ]:
encoding = tiktoken.get_encoding("o200k_base")

def count_tokens(text: str) -> int:
    return len(encoding.encode(str(text)))

def estimate_usage(results: pd.DataFrame) -> pd.DataFrame:
    results = results.copy()
    results["estimated_input_tokens"] = results.apply(
        lambda row: count_tokens(build_user_prompt(row["question"], row["context"])),
        axis=1
    )
    results["estimated_output_tokens"] = results["predicted_answer"].apply(count_tokens)
    return results

results_a = estimate_usage(results_a)
results_b = estimate_usage(results_b)

In [ ]:
PRICING_PER_1M = {
    MODEL_A: {"input": 4.00, "output": 20.00},
    MODEL_B: {"input": 0.20, "output": 1.20},
}

def calculate_cost(results: pd.DataFrame, model_name: str) -> dict:
    input_tokens = results["estimated_input_tokens"].sum()
    output_tokens = results["estimated_output_tokens"].sum()

    input_cost = input_tokens / 1_000_000 * PRICING_PER_1M[model_name]["input"]
    output_cost = output_tokens / 1_000_000 * PRICING_PER_1M[model_name]["output"]

    return {
        "Model": model_name,
        "Input Tokens": f"{input_tokens:,}",
        "Output Tokens": f"{output_tokens:,}",
        "Input Cost": f"${input_cost:.4f}",
        "Output Cost": f"${output_cost:.4f}",
        "Total Cost": f"${input_cost + output_cost:.4f}",
    }

cost_comparison = pd.DataFrame([
    calculate_cost(results_a, MODEL_A),
    calculate_cost(results_b, MODEL_B)
])

cost_comparison

,Model,Input Tokens,Output Tokens,Input Cost,Output Cost,Total Cost
0,gpt-5.6-sol,"5,209","1,252",$0.0208,$0.0250,$0.0459
1,gpt-5.6-luna,"5,209","1,134",$0.0010,$0.0014,$0.0024


In [ ]:
final_summary = pd.DataFrame({
    "Dimension": [
        "Accuracy (post-HITL)",
        "Answer Accuracy",
        "Abstention Rate",
        "Coverage",
        "Exact Match Rate",
        "Cross-Model Agreement",
        "Estimated Cost (50 samples)",
    ],
    MODEL_A: [
        f"{metrics_a_final['Accuracy']:.1%}",
        f"{metrics_a_final['Answer Accuracy']:.1%}",
        f"{metrics_a_final['Abstention Rate']:.1%}",
        f"{metrics_a_final['Coverage']:.1%}",
        f"{results_a['exact_match'].mean():.1%}",
        f"{agreement.mean():.1%}",
        f"${float(cost_comparison[cost_comparison['Model']==MODEL_A]['Total Cost'].values[0].replace('$','')):.4f}",
    ],
    MODEL_B: [
        f"{metrics_b_final['Accuracy']:.1%}",
        f"{metrics_b_final['Answer Accuracy']:.1%}",
        f"{metrics_b_final['Abstention Rate']:.1%}",
        f"{metrics_b_final['Coverage']:.1%}",
        f"{results_b['exact_match'].mean():.1%}",
        f"{agreement.mean():.1%}",
        f"${float(cost_comparison[cost_comparison['Model']==MODEL_B]['Total Cost'].values[0].replace('$','')):.4f}",
    ],
})

final_summary

,Dimension,gpt-5.6-sol,gpt-5.6-luna
0,Accuracy (post-HITL),100.0%,100.0%
1,Answer Accuracy,100.0%,100.0%
2,Abstention Rate,0.0%,0.0%
3,Coverage,100.0%,100.0%
4,Exact Match Rate,8.0%,8.0%
5,Cross-Model Agreement,28.0%,28.0%
6,Estimated Cost (50 samples),$0.0459,$0.0024


In [ ]:
results_a.head(5)

,sample_id,question,context,reference_answer,predicted_answer,confidence,abstained,response_id,error,model,exact_match,judge_verdict,estimated_input_tokens,estimated_output_tokens
0,1,What area did NVIDIA initially focus on before...,"Since our original focus on PC graphics, we ha...",NVIDIA initially focused on PC graphics.,PC graphics,1.0,False,resp_0d4cc72002736cfa006a9ea8e5b8b887d2a6cee4d...,None,gpt-5.6-sol,False,CORRECT,68,2
1,2,What are some of the recent applications of GP...,Some of the most recent applications of GPU-po...,Recent applications of GPU-powered deep learni...,Recent applications include recommendation sys...,1.0,False,resp_008500b76fba7dc7006a9ea8e8353487d298bc79e...,None,gpt-5.6-sol,False,CORRECT,152,15
2,3,What significant invention did NVIDIA create i...,Our invention of the GPU in 1999 defined moder...,NVIDIA invented the GPU in 1999.,The GPU (graphics processing unit),1.0,False,resp_0cc6134ff59a9f38006a9ea8eac68c87d2bbc606b...,None,gpt-5.6-sol,False,CORRECT,64,7
3,4,How does NVIDIA's platform strategy contribute...,"NVIDIA has a platform strategy, bringing toget...",NVIDIA's platform strategy brings together har...,NVIDIA’s platform strategy integrates hardware...,1.0,False,resp_02abef6fac77dbb1006a9ea8ee233087d2a1dc6a6...,None,gpt-5.6-sol,False,CORRECT,74,28
4,5,What does NVIDIA's CUDA programming model enable?,With our introduction of the CUDA programming ...,NVIDIA's CUDA programming model opened the par...,It enables the GPU’s parallel processing capab...,1.0,False,resp_0ad69bb64be642ec006a9ea8efecd487d293448ef...,None,gpt-5.6-sol,False,CORRECT,66,16


In [163]:
results_b.head(5)

,sample_id,question,context,reference_answer,predicted_answer,confidence,abstained,response_id,error,model,exact_match,judge_verdict,estimated_input_tokens,estimated_output_tokens
0,1,What area did NVIDIA initially focus on before...,"Since our original focus on PC graphics, we ha...",NVIDIA initially focused on PC graphics.,PC graphics,1.0,False,resp_071e93c5774104e7006a9ea95cf1c887d28784bd1...,None,gpt-5.6-luna,False,CORRECT,68,2
1,2,What are some of the recent applications of GP...,Some of the most recent applications of GPU-po...,Recent applications of GPU-powered deep learni...,"Recommendation systems, large language models,...",1.0,False,resp_0135d5cd7f8574bb006a9ea95e51b087d2ae9cf11...,None,gpt-5.6-luna,False,CORRECT,152,12
2,3,What significant invention did NVIDIA create i...,Our invention of the GPU in 1999 defined moder...,NVIDIA invented the GPU in 1999.,The GPU (graphics processing unit).,1.0,False,resp_063d347ab16edeca006a9ea964033887d289f4c25...,None,gpt-5.6-luna,False,CORRECT,64,7
3,4,How does NVIDIA's platform strategy contribute...,"NVIDIA has a platform strategy, bringing toget...",NVIDIA's platform strategy brings together har...,"NVIDIA’s platform strategy combines hardware, ...",1.0,False,resp_09b7db68c77b5675006a9ea966bf8c87d2902233c...,None,gpt-5.6-luna,False,CORRECT,74,28
4,5,What does NVIDIA's CUDA programming model enable?,With our introduction of the CUDA programming ...,NVIDIA's CUDA programming model opened the par...,It enables general-purpose computing using the...,1.0,False,resp_0ec7dcaa34d47bf3006a9ea96832c887d2819158b...,None,gpt-5.6-luna,False,CORRECT,66,15


## Recommendation

Both GPT-5.6 Sol and GPT-5.6 Luna were evaluated on the same 50 Financial Q&A 10-K samples using an identical prompt structure and supporting context. Based on the final evaluation (including HITL corrections), the two models demonstrated comparable answer quality on this dataset; although given that it was on an extremely small dataset I would be keen on carrying out Evals on a minimum of 100,000 documents; however that is a topic for another time.

### Production Candidate: GPT-5.6 Luna

Given comparable validated performance, **GPT-5.6 Luna** is the recommended initial production candidate. It provides substantially lower inference cost ; a critical consideration for a financial QA application operating at scale, particularly when there is no observed quality advantage from the more expensive alternative.

### Limitations & Caveats

This recommendation is subject to further validation before production deployment:

- The evaluation covers only **50 samples**, providing limited evidence of generalisation across a broader financial QA workload
- A production evaluation should include a larger, more representative test set covering:
  - Complex numerical reasoning questions
  - Ambiguous or underspecified questions
  - Unsupported questions requiring abstention
  - Diverse companies, sectors, and filing types

### Monitoring in Production

The following metrics should be tracked during any production pilot:

| Metric | Purpose |
|---|---|
| Answer accuracy & semantic correctness | Core quality signal |
| Abstention rate | Uncertainty calibration |
| Hallucination / unsupported-answer rate | Safety and trust |
| Consistency across similar questions | Reliability |
| Latency & throughput | Operational performance |
| Cost per query | Economic efficiency |

### Final Takeaway

> The final production decision should be based on whether Luna continues to match Sol's quality on a larger evaluation set. If it does, Luna is the clear choice due to its cost advantage. If Sol demonstrates a meaningful, repeatable quality edge on more challenging examples, the additional cost may be justified.
>
> **Model selection should not be based on answer quality alone.** For a production financial QA system, the appropriate choice is the model that provides the best balance of accuracy, reliability, abstention behaviour, latency, and cost.